In [0]:
from pyspark.sql import DataFrame, functions as F

SILVER_TABLE = "stryktipset.silver.matches"
DIM_TEAM_TABLE = "stryktipset.gold.dim_team"
DIM_DATE_TABLE = "stryktipset.gold.dim_date"
DIM_LEAGUE_TABLE = "stryktipset.gold.dim_league"
DIM_SEASON_TABLE = "stryktipset.gold.dim_season"

LEAGUE_COUNTRY = {
    "Premier League": "England",
    "Championship": "England",
    "League One": "England",
    "League Two": "England",
    "National League": "England",
    "Allsvenskan": "Sweden",
    "Superettan": "Sweden",
    "Ettan Norra": "Sweden",
    "Ettan Södra": "Sweden",
    "Div 2, Norra Götaland": "Sweden",
    "Div 2, Norra Svealand": "Sweden",
    "Div 2, Södra Götaland": "Sweden",
    "Div 2, Södra Svealand": "Sweden",
    "Div 2, Västra Götaland": "Sweden",
}

In [0]:
def build_dim_team(df: DataFrame) -> DataFrame:

    home_team = df.select(F.col('home_team').alias('team_name'))
    away_team = df.select(F.col('away_team').alias('team_name'))

    return (
        home_team
        .union(away_team)
        .withColumn('team_key', F.upper('team_name'))
        .dropDuplicates(['team_key'])
    )

In [0]:
import datetime

def build_dim_date(df: DataFrame) -> DataFrame:
    """Full calendar, one row per day -- not just dates that had matches.
    Spans from the earliest match in your data to two years past today,
    so upcoming draws are already covered without ever needing to extend
    this table by hand."""
    min_date = df.agg(F.min(F.to_date("match_start"))).first()[0]
    max_date = datetime.date.today() + datetime.timedelta(days=365 * 2)

    return (
        spark.createDataFrame([(min_date, max_date)], ["start", "end"])
        .select(F.explode(F.sequence("start", "end", F.expr("interval 1 day"))).alias("full_date"))
        .withColumn("date_key", F.date_format("full_date", "yyyyMMdd").cast("int"))
        .withColumn("year", F.year("full_date"))
        .withColumn("quarter", F.quarter("full_date"))
        .withColumn("month", F.month("full_date"))
        .withColumn("month_name", F.date_format("full_date", "MMMM"))
        .withColumn("day", F.dayofmonth("full_date"))
        .withColumn("day_of_week", F.date_format("full_date", "u").cast("int"))  # 1=Monday..7=Sunday
        .withColumn("day_name", F.date_format("full_date", "EEEE"))
        .withColumn("week_of_year", F.weekofyear("full_date"))
        .withColumn("is_weekend", F.dayofweek("full_date").isin(6, 7))
    )

In [0]:
def build_dim_league(league_country: dict = LEAGUE_COUNTRY) -> DataFrame:
    """One row per key in LEAGUE_COUNTRY, with its country. Built from
    this dict, not from matches -- any league not in LEAGUE_COUNTRY
    simply doesn't get a row, and (via the filter in 04) won't appear in
    fact_match either.

    Columns: league_key, league_name, country.
    """
    # TODO

In [0]:
def build_dim_season(df: DataFrame) -> DataFrame:
    """One row per football season, derived from match_start. Seasons
    span calendar years -- something like "if month >= July, season is
    year/year+1, else year-1/year".

    Columns: season_key, season_label.
    """
    # TODO

In [0]:
def write_dimension(df: DataFrame, table: str) -> None:
    """Full overwrite, same as 02_silver_transform does with matches --
    no incremental merge needed."""
    # TODO

In [0]:
def main():
    df = spark.table(SILVER_TABLE)

    write_dimension(build_dim_team(df), DIM_TEAM_TABLE)
    write_dimension(build_dim_date(df), DIM_DATE_TABLE)
    write_dimension(build_dim_league(), DIM_LEAGUE_TABLE)
    write_dimension(build_dim_season(df), DIM_SEASON_TABLE)

main()